# Проект по биоинформатике. Anopheles stephensi

# Импорты и загрузки

In [1]:
!pip install biopython
!pip install fuc
!apt-get install bedtools

from Bio import SeqIO
from Bio.Seq import Seq
from Bio.Data import CodonTable
from Bio import Entrez
from Bio.SeqFeature import SeqFeature, FeatureLocation
import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 80.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 68.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 88.2 MB/s eta 0:00:00
  Created wheel for sorted_nearest: filename=sorted_nearest-0.0.41-cp312-cp312-linux_x86_64.whl size=6690263 sha256=4a55fda4c6847b7cd56947976e7987cae6587324e76d2ade01c8f659b37f2e4e
  Stored in directory: /root/.cache/pip/wheels/7c/cb/a4/a6f831a60e479b9001be190e91000cc472addeab871957e5a9
Successfully built sorted_nearest
Reading package lists... Done
Building dependency tree... Done
Reading st

In [2]:
!gdown "1lqiyL7d9Zu4ztkCTwpJKVfSwqoJmAp-b" # genomic.fna


scaffolds = {}
for scaffold in SeqIO.parse('GCF_013141755.1_UCI_ANSTEP_V1.0_genomic.fna', format='fasta'):
    scaffolds[scaffold.id] = scaffold
len(scaffolds)

Downloading...
From (original): https://drive.google.com/uc?id=1lqiyL7d9Zu4ztkCTwpJKVfSwqoJmAp-b
From (redirected): https://drive.google.com/uc?id=1lqiyL7d9Zu4ztkCTwpJKVfSwqoJmAp-b&confirm=t&uuid=379554fa-8464-4172-9c33-b1d197cb81b0
To: /content/GCF_013141755.1_UCI_ANSTEP_V1.0_genomic.fna
100% 247M/247M [00:05<00:00, 47.0MB/s]


494

Протеомы

In [3]:
!gdown "19TECLovpngLrwcamfVbsX5kncOfs4htI"    # genomic.gff

annotation_df = pd.read_csv(
    'genomic.gff',
    sep='\t',
    comment='#',
    header=None,
    names=["scaffold", "source", "type", "start", "end", "score", "strand", "phase", "attributes"]
)

cds_df = annotation_df[annotation_df['type'] == 'CDS'].reset_index()
print(f"Найдено {len(cds_df)} CDS")

Downloading...
From (original): https://drive.google.com/uc?id=19TECLovpngLrwcamfVbsX5kncOfs4htI
From (redirected): https://drive.google.com/uc?id=19TECLovpngLrwcamfVbsX5kncOfs4htI&confirm=t&uuid=d884597f-f929-455f-8918-0691150bcc60
To: /content/genomic.gff
100% 132M/132M [00:02<00:00, 57.7MB/s]
Найдено 191820 CDS


In [4]:
with open('proteins.fasta', 'w') as prot_file:
    cur_transcript=None
    cur_cds = Seq("")
    for index, row in cds_df.iterrows():
        attrs = row['attributes'].split(';')
        parent = attrs[1]
        gene = attrs[5]
        phase = 0
        if parent != cur_transcript:
            if index != 0:
                if strand == '-':
                    cur_cds = cur_cds.reverse_complement()
                protein_seq = cur_cds.translate(to_stop=True)
                prot_file.write(f'>{gene} scaffold={scaffold} strand={strand} {cur_transcript} \n')
                prot_file.write(f'{str(protein_seq)}\n')

            cur_transcript = parent
            cur_cds = Seq("")
            strand = row['strand']
            phase = int(row['phase'])
            scaffold = row['scaffold']

        beg = row['start'] - 1 + phase
        end = row['end']
        seq = scaffolds[scaffold].seq[beg:end]
        cur_cds = cur_cds + seq

    if strand == '-':
        cur_cds = cur_cds.reverse_complement()
    protein_seq = cur_cds.translate(to_stop=True)
    prot_file.write(f'> {scaffold} {strand} {cur_transcript}\n')
    prot_file.write(f'{str(protein_seq)}\n')

/usr/local/lib/python3.12/dist-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


# 1. Проверка генов на эпигенетику

Подготовка

In [5]:
%%bash
wget http://eddylab.org/software/hmmer/hmmer-3.3.2.tar.gz
tar xf hmmer-3.3.2.tar.gz
cd hmmer-3.3.2
./configure
make
make install

configure: Configuring HMMER3 for your system.
checking build system type... x86_64-pc-linux-gnu
checking host system type... x86_64-pc-linux-gnu
checking whether to compile using MPI... no
checking for gcc... gcc
checking whether the C compiler works... yes
checking for C compiler default output file name... a.out
checking for suffix of executables... 
checking whether we are cross compiling... no
checking for suffix of object files... o
checking whether we are using the GNU C compiler... yes
checking whether gcc accepts -g... yes
checking for gcc option to accept ISO C89... none needed
checking for gcc option to accept ISO C99... none needed
checking for gcc option to accept ISO Standard C... (cached) none needed
checking how to run the C preprocessor... gcc -E
checking for a BSD-compatible install... /usr/bin/install -c
checking for strip... strip
checking for ranlib... ranlib
checking for ar... /usr/bin/ar
checking whether ln -s works... yes
checking for a sed that does not truncat

--2026-06-09 18:12:49--  http://eddylab.org/software/hmmer/hmmer-3.3.2.tar.gz
Resolving eddylab.org (eddylab.org)... 96.126.110.11, 2600:3c03::f03c:91ff:fec8:383c
Connecting to eddylab.org (eddylab.org)|96.126.110.11|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 18213049 (17M) [application/x-gzip]
Saving to: ‘hmmer-3.3.2.tar.gz’

     0K .......... .......... .......... .......... ..........  0%  111K 2m39s
    50K .......... .......... .......... .......... ..........  0%  223K 1m59s
   100K .......... .......... .......... .......... ..........  0%  125M 79s
   150K .......... .......... .......... .......... ..........  1% 37.9M 59s
   200K .......... .......... .......... .......... ..........  1%  224K 63s
   250K .......... .......... .......... .......... ..........  1% 55.0M 52s
   300K .......... .......... .......... .......... ..........  1% 78.0M 45s
   350K .......... .......... .......... .......... ..........  2%  221M 39s
   400K .......... ..

In [6]:
!wget http://ftp.ebi.ac.uk/pub/databases/Pfam/releases/Pfam35.0/Pfam-A.hmm.gz

--2026-06-09 18:14:22--  http://ftp.ebi.ac.uk/pub/databases/Pfam/releases/Pfam35.0/Pfam-A.hmm.gz
Resolving ftp.ebi.ac.uk (ftp.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.ebi.ac.uk (ftp.ebi.ac.uk)|193.62.193.165|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 293000230 (279M) [application/x-gzip]
Saving to: ‘Pfam-A.hmm.gz’

Pfam-A.hmm.gz       100%[===================>] 279.43M  3.49MB/s    in 6m 8s   

2026-06-09 18:20:31 (778 KB/s) - ‘Pfam-A.hmm.gz’ saved [293000230/293000230]



In [7]:
!gunzip Pfam-A.hmm.gz

In [8]:
!hmmpress Pfam-A.hmm

Working...    done.
Pressed and indexed 19632 HMMs (19632 names and 19632 accessions).
Models pressed into binary file:   Pfam-A.hmm.h3m
SSI index for binary model file:   Pfam-A.hmm.h3i
Profiles (MSV part) pressed into:  Pfam-A.hmm.h3f
Profiles (remainder) pressed into: Pfam-A.hmm.h3p


In [9]:
!head -n 40 Pfam-A.hmm

HMMER3/f [3.1b2 | February 2015]
NAME  1-cysPrx_C
ACC   PF10417.12
DESC  C-terminal domain of 1-Cys peroxiredoxin
LENG  40
ALPH  amino
RF    no
MM    no
CONS  yes
CS    yes
MAP   yes
DATE  Thu Nov  4 19:20:02 2021
NSEQ  40
EFFN  17.426758
CKSUM 4086680297
GA    21.1 21.1;
TC    21.1 21.1;
NC    21 21;
BM    hmmbuild HMM.ann SEED.ann
SM    hmmsearch -Z 61295632 -E 1000 --cpu 4 HMM pfamseq
STATS LOCAL MSV       -7.5463  0.71948
STATS LOCAL VITERBI   -7.8624  0.71948
STATS LOCAL FORWARD   -4.3303  0.71948
HMM          A        C        D        E        F        G        H        I        K        L        M        N        P        Q        R        S        T        V        W        Y   
            m->m     m->i     m->d     i->m     i->i     d->m     d->d
  COMPO   2.28046  4.31208  2.83393  2.63913  3.90855  2.69988  3.89812  3.33401  2.56310  2.85023  3.99954  3.22924  2.52123  2.90328  3.31238  2.94055  2.70512  2.59551  3.49266  3.82715
          2.68618  4.42225  2.77519  2.7312

In [10]:
!hmmstat Pfam-A.hmm.h3m | head -n 20

# hmmstat :: display summary statistics for a profile file
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
#
# idx  name                 accession        nseq eff_nseq      M relent   info p relE compKL
# ---- -------------------- ------------ -------- -------- ------ ------ ------ ------ ------
1      1-cysPrx_C           PF10417.12         40    17.43     40   1.37   1.32   1.29   0.09
2      120_Rick_ant         PF12574.11          2     0.45    238   0.59   0.60   0.52   0.02
3      12TM_1               PF09847.12          7     2.28    449   0.59   0.62   0.51   0.09
4      14-3-3               PF00244.23        149     2.50    222   0.59   0.59   0.52   0.05
5      17kDa_Anti_2         PF16998.8           6     0.99    116   0.59   0.57   0.52   0.03
6      2-Hacid_dh           PF00389.33         78    

In [11]:
!hmmfetch --index Pfam-A.hmm.h3m

Working...    done.
Indexed 19632 HMMs (19632 names and 19632 accessions).
SSI index written to file Pfam-A.hmm.h3m.ssi


# Выбор и анализ 10 семейств

Выбрал следующие: ALKBH1, BAP1, HDAC11, EEF1AKMT4, SUDS3, SIRT2, ZNHIT1, SAP18, PRMT8, TDG

# 1) ALKBH1

In [12]:
!hmmfetch Pfam-A.hmm.h3m PF13532.9 > alkbh1.hmm

In [13]:
!grep PF13532 Pfam-A.hmm

ACC   PF13532.9


In [14]:
!hmmstat alkbh1.hmm

# hmmstat :: display summary statistics for a profile file
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
#
# idx  name                 accession        nseq eff_nseq      M relent   info p relE compKL
# ---- -------------------- ------------ -------- -------- ------ ------ ------ ------ ------
1      2OG-FeII_Oxy_2       PF13532.9          36     4.21    194   0.59   0.55   0.47   0.02


In [15]:
!hmmsearch alkbh1.hmm proteins.fasta

# hmmsearch :: search profile(s) against a sequence database
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
# query HMM file:                  alkbh1.hmm
# target sequence database:        proteins.fasta
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

Query:       2OG-FeII_Oxy_2  [M=194]
Accession:   PF13532.9
Description: 2OG-Fe(II) oxygenase superfamily
Scores for complete sequences (score includes all domains):
   --- full sequence ---   --- best 1 domain ---    -#dom-
    E-value  score  bias    E-value  score  bias    exp  N  Sequence          Description
    ------- ------ -----    ------- ------ -----   ---- --  --------          -----------
    4.2e-23   83.2   0.0    8.4e-23   82.2   0.0    1.5  1  gene=LOC118508064  scaffold=NC_050202.1 strand=+ Parent=rna-X
      2e-14  

# 2) BAP1

In [16]:
!grep PF01088 Pfam-A.hmm

ACC   PF01088.24


In [17]:
!hmmfetch Pfam-A.hmm.h3m PF01088.24 > bap1.hmm
!hmmstat bap1.hmm

# hmmstat :: display summary statistics for a profile file
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
#
# idx  name                 accession        nseq eff_nseq      M relent   info p relE compKL
# ---- -------------------- ------------ -------- -------- ------ ------ ------ ------ ------
1      Peptidase_C12        PF01088.24        349     2.96    211   0.59   0.57   0.44   0.02


In [18]:
!hmmsearch bap1.hmm proteins.fasta

# hmmsearch :: search profile(s) against a sequence database
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
# query HMM file:                  bap1.hmm
# target sequence database:        proteins.fasta
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

Query:       Peptidase_C12  [M=211]
Accession:   PF01088.24
Description: Ubiquitin carboxyl-terminal hydrolase, family 1
Scores for complete sequences (score includes all domains):
   --- full sequence ---   --- best 1 domain ---    -#dom-
    E-value  score  bias    E-value  score  bias    exp  N  Sequence          Description
    ------- ------ -----    ------- ------ -----   ---- --  --------          -----------
    8.3e-70  235.5   0.0    1.3e-69  234.9   0.0    1.3  1  gene=LOC118512888  scaffold=NC_050203.1 strand=+ Parent=rna-X


# 3) HDAC11

In [19]:
!grep PF00850 Pfam-A.hmm

ACC   PF00850.22


In [20]:
!hmmfetch Pfam-A.hmm.h3m PF00850.22 > HDAC11.hmm
!hmmstat HDAC11.hmm

# hmmstat :: display summary statistics for a profile file
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
#
# idx  name                 accession        nseq eff_nseq      M relent   info p relE compKL
# ---- -------------------- ------------ -------- -------- ------ ------ ------ ------ ------
1      Hist_deacetyl        PF00850.22         68     2.85    307   0.59   0.57   0.48   0.02


In [21]:
!hmmsearch HDAC11.hmm proteins.fasta

# hmmsearch :: search profile(s) against a sequence database
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
# query HMM file:                  HDAC11.hmm
# target sequence database:        proteins.fasta
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

Query:       Hist_deacetyl  [M=307]
Accession:   PF00850.22
Description: Histone deacetylase domain
Scores for complete sequences (score includes all domains):
   --- full sequence ---   --- best 1 domain ---    -#dom-
    E-value  score  bias    E-value  score  bias    exp  N  Sequence          Description
    ------- ------ -----    ------- ------ -----   ---- --  --------          -----------
    6.2e-89  299.1   0.1    8.1e-89  298.7   0.1    1.1  1  gene=LOC118511268  scaffold=NC_050203.1 strand=+ Parent=rna-X
    2.7e-88  297.0 

# 4) EEF1AKMT4

In [22]:
!grep PF08241 Pfam-A.hmm

ACC   PF08241.15


In [23]:
!hmmfetch Pfam-A.hmm.h3m PF08241.15 > EEF1AKMT4.hmm
!hmmstat EEF1AKMT4.hmm

# hmmstat :: display summary statistics for a profile file
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
#
# idx  name                 accession        nseq eff_nseq      M relent   info p relE compKL
# ---- -------------------- ------------ -------- -------- ------ ------ ------ ------ ------
1      Methyltransf_11      PF08241.15        141     7.11     96   0.60   0.61   0.48   0.01


In [24]:
!hmmsearch EEF1AKMT4.hmm proteins.fasta

# hmmsearch :: search profile(s) against a sequence database
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
# query HMM file:                  EEF1AKMT4.hmm
# target sequence database:        proteins.fasta
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

Query:       Methyltransf_11  [M=96]
Accession:   PF08241.15
Description: Methyltransferase domain
Scores for complete sequences (score includes all domains):
   --- full sequence ---   --- best 1 domain ---    -#dom-
    E-value  score  bias    E-value  score  bias    exp  N  Sequence               Description
    ------- ------ -----    ------- ------ -----   ---- --  --------               -----------
    1.7e-20   74.1   0.0    2.8e-20   73.5   0.0    1.3  1  gene=LOC118506044       scaffold=NC_050202.1 strand=+ Parent=
    1.1

# 5) SUDS3

In [25]:
!grep PF08598 Pfam-A.hmm

ACC   PF08598.14


In [26]:
!hmmfetch Pfam-A.hmm.h3m PF08598.14 > SUDS3.hmm
!hmmstat SUDS3.hmm

# hmmstat :: display summary statistics for a profile file
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
#
# idx  name                 accession        nseq eff_nseq      M relent   info p relE compKL
# ---- -------------------- ------------ -------- -------- ------ ------ ------ ------ ------
1      Sds3                 PF08598.14         59     4.57    218   0.59   0.59   0.43   0.07


In [27]:
!hmmsearch SUDS3.hmm proteins.fasta

# hmmsearch :: search profile(s) against a sequence database
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
# query HMM file:                  SUDS3.hmm
# target sequence database:        proteins.fasta
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

Query:       Sds3  [M=218]
Accession:   PF08598.14
Description: Sds3-like
Scores for complete sequences (score includes all domains):
   --- full sequence ---   --- best 1 domain ---    -#dom-
    E-value  score  bias    E-value  score  bias    exp  N  Sequence          Description
    ------- ------ -----    ------- ------ -----   ---- --  --------          -----------
    1.2e-18   68.7   6.5    4.5e-18   66.8   6.5    1.7  1  gene=LOC118505026  scaffold=NC_050202.1 strand=+ Parent=rna-X
    4.8e-17   63.4   1.7    6.9e-17   62.9   1

# 6) SIRT2

In [28]:
!grep PF02146 Pfam-A.hmm

ACC   PF02146.20


In [29]:
!hmmfetch Pfam-A.hmm.h3m PF02146.20 > SIRT2.hmm
!hmmstat SIRT2.hmm

# hmmstat :: display summary statistics for a profile file
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
#
# idx  name                 accession        nseq eff_nseq      M relent   info p relE compKL
# ---- -------------------- ------------ -------- -------- ------ ------ ------ ------ ------
1      SIR2                 PF02146.20         18     1.22    179   0.59   0.56   0.51   0.01


In [30]:
!hmmsearch SIRT2.hmm proteins.fasta

# hmmsearch :: search profile(s) against a sequence database
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
# query HMM file:                  SIRT2.hmm
# target sequence database:        proteins.fasta
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

Query:       SIR2  [M=179]
Accession:   PF02146.20
Description: Sir2 family
Scores for complete sequences (score includes all domains):
   --- full sequence ---   --- best 1 domain ---    -#dom-
    E-value  score  bias    E-value  score  bias    exp  N  Sequence          Description
    ------- ------ -----    ------- ------ -----   ---- --  --------          -----------
    1.3e-57  195.4   0.0    2.5e-57  194.4   0.0    1.5  1  gene=LOC118504478  scaffold=NC_050202.1 strand=+ Parent=rna-X
    1.4e-45  156.1   0.0    2.4e-45  155.4  

# 7) ZNHIT1

In [31]:
!grep PF04438 Pfam-A.hmm

ACC   PF04438.19


In [32]:
!hmmfetch Pfam-A.hmm.h3m PF04438.19 > ZNHIT1.hmm
!hmmstat ZNHIT1.hmm

# hmmstat :: display summary statistics for a profile file
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
#
# idx  name                 accession        nseq eff_nseq      M relent   info p relE compKL
# ---- -------------------- ------------ -------- -------- ------ ------ ------ ------ ------
1      zf-HIT               PF04438.19         87    12.24     30   1.80   1.43   1.72   0.57


In [33]:
!hmmsearch ZNHIT1.hmm proteins.fasta

# hmmsearch :: search profile(s) against a sequence database
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
# query HMM file:                  ZNHIT1.hmm
# target sequence database:        proteins.fasta
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

Query:       zf-HIT  [M=30]
Accession:   PF04438.19
Description: HIT zinc finger
Scores for complete sequences (score includes all domains):
   --- full sequence ---   --- best 1 domain ---    -#dom-
    E-value  score  bias    E-value  score  bias    exp  N  Sequence          Description
    ------- ------ -----    ------- ------ -----   ---- --  --------          -----------
    1.3e-10   41.6  13.3    2.6e-10   40.7  13.3    1.5  1  gene=LOC118514154  scaffold=NC_050203.1 strand=- Parent=rna-X
    4.2e-09   36.8  10.0    9.4e-09   

# 8) SAP18

In [34]:
!grep PF06487 Pfam-A.hmm

ACC   PF06487.15


In [35]:
!hmmfetch Pfam-A.hmm.h3m PF06487.15 > SAP18.hmm
!hmmstat SAP18.hmm

# hmmstat :: display summary statistics for a profile file
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
#
# idx  name                 accession        nseq eff_nseq      M relent   info p relE compKL
# ---- -------------------- ------------ -------- -------- ------ ------ ------ ------ ------
1      SAP18                PF06487.15        111     2.70    141   0.59   0.58   0.46   0.02


In [36]:
!hmmsearch SAP18.hmm proteins.fasta

# hmmsearch :: search profile(s) against a sequence database
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
# query HMM file:                  SAP18.hmm
# target sequence database:        proteins.fasta
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

Query:       SAP18  [M=141]
Accession:   PF06487.15
Description: Sin3 associated polypeptide p18 (SAP18)
Scores for complete sequences (score includes all domains):
   --- full sequence ---   --- best 1 domain ---    -#dom-
    E-value  score  bias    E-value  score  bias    exp  N  Sequence          Description
    ------- ------ -----    ------- ------ -----   ---- --  --------          -----------
      8e-37  127.3   0.0      1e-36  126.9   0.0    1.2  1  gene=LOC118506822  scaffold=NC_050202.1 strand=- Parent=rna-X


Domain annota

# 9)PRMT8

In [37]:
!grep PF06325 Pfam-A.hmm

ACC   PF06325.16


In [38]:
!hmmfetch Pfam-A.hmm.h3m PF06325.16 > PRMT8.hmm
!hmmstat PRMT8.hmm

# hmmstat :: display summary statistics for a profile file
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
#
# idx  name                 accession        nseq eff_nseq      M relent   info p relE compKL
# ---- -------------------- ------------ -------- -------- ------ ------ ------ ------ ------
1      PrmA                 PF06325.16         27     1.53    295   0.59   0.59   0.51   0.02


In [39]:
!hmmsearch PRMT8.hmm proteins.fasta

# hmmsearch :: search profile(s) against a sequence database
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
# query HMM file:                  PRMT8.hmm
# target sequence database:        proteins.fasta
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

Query:       PrmA  [M=295]
Accession:   PF06325.16
Description: Ribosomal protein L11 methyltransferase (PrmA)
Scores for complete sequences (score includes all domains):
   --- full sequence ---   --- best 1 domain ---    -#dom-
    E-value  score  bias    E-value  score  bias    exp  N  Sequence          Description
    ------- ------ -----    ------- ------ -----   ---- --  --------          -----------
    8.9e-14   52.3   0.7    1.5e-13   51.6   0.7    1.3  1  gene=LOC118508889  scaffold=NC_050203.1 strand=+ Parent=rna-X
    5.7e-

#10) TDG

In [40]:
!grep PF03167 Pfam-A.hmm

ACC   PF03167.22


In [41]:
!hmmfetch Pfam-A.hmm.h3m PF03167.22 > TDG.hmm
!hmmstat TGD.hmm

# hmmstat :: display summary statistics for a profile file
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

Error: File existence/permissions problem in trying to open HMM file TGD.hmm.
HMM file TGD.hmm not found (nor an .h3m binary of it)



In [42]:
!hmmsearch TDG.hmm proteins.fasta

# hmmsearch :: search profile(s) against a sequence database
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
# query HMM file:                  TDG.hmm
# target sequence database:        proteins.fasta
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

Query:       UDG  [M=155]
Accession:   PF03167.22
Description: Uracil DNA glycosylase superfamily
Scores for complete sequences (score includes all domains):
   --- full sequence ---   --- best 1 domain ---    -#dom-
    E-value  score  bias    E-value  score  bias    exp  N  Sequence          Description
    ------- ------ -----    ------- ------ -----   ---- --  --------          -----------
    2.9e-07   31.3   0.0    3.7e-07   31.0   0.0    1.1  1  gene=LOC118504424  scaffold=NC_050202.1 strand=- Parent=rna-X


Domain annotation for 

In [43]:
families = {
    'ALKBH1': 'alkbh1.hmm',
    'BAP1': 'bap1.hmm',
    'HDAC11': 'HDAC11.hmm',
    'EEF1AKMT4': 'EEF1AKMT4.hmm',
    'SUDS3': 'SUDS3.hmm',
    'SIRT2': 'SIRT2.hmm',
    'ZNHIT1': 'ZNHIT1.hmm',
    'SAP18': 'SAP18.hmm',
    'PRMT8': 'PRMT8.hmm',
    'TDG': 'TDG.hmm'
}

results = []

for family, hmm_file in families.items():
    !hmmsearch --tblout temp_output.txt {hmm_file} proteins.fasta

    with open('temp_output.txt', 'r') as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            parts = line.split()
            if len(parts) >= 3 and not parts[0].startswith('#'):
                gene_name = parts[0]
                results.append({
                    'Проверяемое семейство': family,
                    'Название гена': gene_name.split('=')[1] if '=' in gene_name else gene_name,
                })

df_results = pd.DataFrame(results)
print(df_results)

df_results.to_csv('epigenetic_genes.csv', index=False)

# hmmsearch :: search profile(s) against a sequence database
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
# query HMM file:                  alkbh1.hmm
# target sequence database:        proteins.fasta
# per-seq hits tabular output:     temp_output.txt
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

Query:       2OG-FeII_Oxy_2  [M=194]
Accession:   PF13532.9
Description: 2OG-Fe(II) oxygenase superfamily
Scores for complete sequences (score includes all domains):
   --- full sequence ---   --- best 1 domain ---    -#dom-
    E-value  score  bias    E-value  score  bias    exp  N  Sequence          Description
    ------- ------ -----    ------- ------ -----   ---- --  --------          -----------
    4.2e-23   83.2   0.0    8.4e-23   82.2   0.0    1.5  1  gene=LOC118508064  scaff

# 2. Поиск квадруплексов и работа с Zhunt

Квадруплексы

In [44]:
%pip install pybedtools

import pybedtools
import pandas as pd
import re

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 97.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pybedtools: filename=pybedtools-0.12.0-cp312-cp312-linux_x86_64.whl size=14340832 sha256=9e70dcbe98fd736f1961421c3949c3935c4fb1aa57e4ce186f4a2637afb84e03
  Stored in directory: /root/.cache/pip/wheels/ac/38/f2/960d79e44a92afc0d34a4727c856ce0149ac23c3dcda174356
Successfully built pybedtools


In [45]:
PQS = []
PQS_minus = []
pattern="(?:G{3,5}[ATGC]{1,7}){3,}G{3,5}"
pattern_minus="(?:C{3,5}[ATGC]{1,7}){3,}C{3,5}"
for scaffold, record in scaffolds.items():
    name, sequence = record.id, str(record.seq)
    PQS += [[name, m.start(),m.end(),m.group(0), name, '+'] for m in re.finditer(pattern,sequence,re.IGNORECASE)]
    PQS_minus += [[name, m.start(),m.end(),m.group(0), name, '-'] for m in re.finditer(pattern_minus,sequence,re.IGNORECASE)]
print(len(PQS))
print(len(PQS_minus))

8698
8620


In [46]:
quadrs = pd.DataFrame(PQS+PQS_minus, columns=['Chromosome', 'Start', 'End', 'Sequence', 'Name', 'Strand'])
quadrs

,Chromosome,Start,End,Sequence,Name,Strand
0,NC_050201.1,133214,133243,GGGGATTTTACGGGATGGGATGATGCGGG,NC_050201.1,+
1,NC_050201.1,134771,134795,ggggggggggagggggagggtggG,NC_050201.1,+
2,NC_050201.1,135062,135081,GGGTAGGGATCGGGTAGGG,NC_050201.1,+
3,NC_050201.1,234992,235010,GGGCTGGGCTGGGGCGGG,NC_050201.1,+
4,NC_050201.1,246043,246060,ggggggggggggggggg,NC_050201.1,+
...,...,...,...,...,...,...
17313,NW_023405470.1,10434,10456,cccagcagaccctcccagACCC,NW_023405470.1,-
17314,NW_023405470.1,33834,33857,cCCCTCCCCTCATCCCCCTGCCC,NW_023405470.1,-
17315,NW_023405470.1,38555,38595,CCCTGGGCCGCCCTGTTTTCCCCCGTGGTTCCCAGTGCCC,NW_023405470.1,-
17316,NW_023405470.1,41582,41609,CCCGAAGAGGCCCACCCAACCCacccc,NW_023405470.1,-


In [47]:
from fuc import pybed
bf = pybed.BedFrame.from_frame(meta=[], data=quadrs)
bf.to_file('quadruplex.bed')

In [48]:
# Загрузка аннотации GFF
!gdown "19TECLovpngLrwcamfVbsX5kncOfs4htI" # genomic.gff

annotation_df = pd.read_csv(
    'genomic.gff',
    sep='\t',
    comment='#',
    header=None,
    names=["scaffold", "source", "type", "start", "end", "score", "strand", "phase", "attributes"]
)

annotation_df.head()

Downloading...
From (original): https://drive.google.com/uc?id=19TECLovpngLrwcamfVbsX5kncOfs4htI
From (redirected): https://drive.google.com/uc?id=19TECLovpngLrwcamfVbsX5kncOfs4htI&confirm=t&uuid=b6bdadb0-e038-4187-8392-e7016efb4064
To: /content/genomic.gff
100% 132M/132M [00:04<00:00, 31.1MB/s]


,scaffold,source,type,start,end,score,strand,phase,attributes
0,NC_050201.1,RefSeq,region,1,22713616,.,+,.,ID=NC_050201.1:1..22713616;Dbxref=taxon:30069;...
1,NC_050201.1,Gnomon,gene,5174,7880,.,-,.,ID=gene-LOC118517478;Dbxref=GeneID:118517478;N...
2,NC_050201.1,Gnomon,lnc_RNA,5174,7880,.,-,.,ID=rna-XR_004908317.1;Parent=gene-LOC118517478...
3,NC_050201.1,Gnomon,exon,7498,7880,.,-,.,ID=exon-XR_004908317.1-1;Parent=rna-XR_0049083...
4,NC_050201.1,Gnomon,exon,6206,7431,.,-,.,ID=exon-XR_004908317.1-2;Parent=rna-XR_0049083...


Разбиение аннотации

In [49]:
with open('scaffolds_sizes.txt', 'w') as out:
    for scaffold in scaffolds:
        scaffold_name = scaffold
        scaffold_length = len(scaffolds[scaffold].seq)
        out.write(f"{scaffold_name}\t{scaffold_length}\n")

In [50]:
def add_introns(annotation_df):
    introns = []
    for gene in annotation_df[annotation_df['type'] == 'gene'].itertuples():
        gene_exons = annotation_df[(annotation_df['type'] == 'exon') &
                                   (annotation_df['scaffold'] == gene.scaffold) &
                                   (annotation_df['start'] >= gene.start) &
                                   (annotation_df['end'] <= gene.end)]
        gene_exons = gene_exons.sort_values(by='start')
        previous_exon_end = None
        for exon in gene_exons.itertuples():
            if previous_exon_end is not None:
                intron_start = previous_exon_end + 1
                intron_end = exon.start - 1
                if intron_start < intron_end:
                    introns.append({
                        "scaffold": gene.scaffold,
                        "source": "predicted",
                        "type": "intron",
                        "start": intron_start,
                        "end": intron_end,
                        "score": ".",
                        "strand": gene.strand,
                        "phase": ".",
                        "attributes": gene.attributes
                    })
            previous_exon_end = exon.end

    introns_df = pd.DataFrame(introns)
    return pd.concat([annotation_df, introns_df], ignore_index=True)

annotation_intron_df = add_introns(annotation_df)

In [51]:
annotation_intron_df

,scaffold,source,type,start,end,score,strand,phase,attributes
0,NC_050201.1,RefSeq,region,1,22713616,.,+,.,ID=NC_050201.1:1..22713616;Dbxref=taxon:30069;...
1,NC_050201.1,Gnomon,gene,5174,7880,.,-,.,ID=gene-LOC118517478;Dbxref=GeneID:118517478;N...
2,NC_050201.1,Gnomon,lnc_RNA,5174,7880,.,-,.,ID=rna-XR_004908317.1;Parent=gene-LOC118517478...
3,NC_050201.1,Gnomon,exon,7498,7880,.,-,.,ID=exon-XR_004908317.1-1;Parent=rna-XR_0049083...
4,NC_050201.1,Gnomon,exon,6206,7431,.,-,.,ID=exon-XR_004908317.1-2;Parent=rna-XR_0049083...
...,...,...,...,...,...,...,...,...,...
533828,NW_023405470.1,predicted,intron,62781,62845,.,-,.,ID=gene-LOC118517349;Dbxref=GeneID:118517349;N...
533829,NW_023405470.1,predicted,intron,64151,64288,.,+,.,ID=gene-LOC118517350;Dbxref=GeneID:118517350;N...
533830,NW_023405470.1,predicted,intron,64419,64478,.,+,.,ID=gene-LOC118517350;Dbxref=GeneID:118517350;N...
533831,NW_023405470.1,predicted,intron,67065,67139,.,-,.,ID=gene-LOC118517348;Dbxref=GeneID:118517348;N...


In [52]:
annotation_intron_df[['scaffold', 'start', 'end', 'type', 'scaffold', 'strand']].to_csv('annotation.bed', header=None, index=False, sep='\t')

In [53]:
!awk '$4 == "exon"' annotation.bed > exons.bed
!awk '$4 == "intron"' annotation.bed > introns.bed
!awk '$4 == "gene"' annotation.bed > genes.bed

!bedtools complement -i genes.bed -g scaffolds_sizes.txt > intergenic.bed
!bedtools flank -i genes.bed -g scaffolds_sizes.txt -l 1000 -r 0 -s > promoters.bed
!bedtools flank -i genes.bed -g scaffolds_sizes.txt -l 0 -r 200 -s > downstream.bed

!bedtools intersect -a quadruplex.bed -b exons.bed -u > quadr_in_exons.bed
!bedtools intersect -a quadruplex.bed -b introns.bed -u > quadr_in_introns.bed
!bedtools intersect -a quadruplex.bed -b promoters.bed -u > quadr_in_promoters.bed
!bedtools intersect -a quadruplex.bed -b downstream.bed -u > quadr_in_downstream.bed
!bedtools intersect -a quadruplex.bed -b intergenic.bed -u > quadr_in_intergenic.bed

In [54]:
!wc -l quadr_in_exons.bed
!wc -l quadr_in_introns.bed
!wc -l quadr_in_promoters.bed
!wc -l quadr_in_downstream.bed
!wc -l quadr_in_intergenic.bed
!wc -l quadruplex.bed

2548 quadr_in_exons.bed
9193 quadr_in_introns.bed
740 quadr_in_promoters.bed
145 quadr_in_downstream.bed
5812 quadr_in_intergenic.bed
17318 quadruplex.bed


In [55]:
!bedtools intersect -a exons.bed -b quadruplex.bed -wa > exons_with_quadr.bed
!wc -l exons_with_quadr.bed
!wc -l exons.bed

8013 exons_with_quadr.bed
229796 exons.bed


In [56]:
%%bash
bedtools intersect -a introns.bed -b quadruplex.bed -wa > introns_with_quadr.bed
wc -l introns_with_quadr.bed
wc -l introns.bed

9603 introns_with_quadr.bed
62211 introns.bed


In [57]:
%%bash
bedtools intersect -a promoters.bed -b quadruplex.bed -wa > promoters_with_quadr.bed
wc -l promoters_with_quadr.bed
wc -l promoters.bed

769 promoters_with_quadr.bed
15186 promoters.bed


In [58]:
%%bash
bedtools intersect -a downstream.bed -b quadruplex.bed -wa > downstream_with_quadr.bed
wc -l downstream_with_quadr.bed
wc -l downstream.bed

151 downstream_with_quadr.bed
15187 downstream.bed


In [59]:
!bedtools intersect -a intergenic.bed -b quadruplex.bed -wa > intergenic_with_quadr.bed
!wc -l intergenic_with_quadr.bed
!wc -l intergenic.bed

5812 intergenic_with_quadr.bed
12343 intergenic.bed


Zhunt

In [60]:
!gcc zhunt.c -lm -o zhunt -D_THREAD_SAFE

cc1: fatal error: zhunt.c: No such file or directory
compilation terminated.


In [61]:
#!./zhunt 12 8 12 GCF_013141755.1_UCI_ANSTEP_V1.0_genomic.fna

In [62]:
!gdown "1N_-vMLwN0t6m4PjEdFgY7VySCiXwC1qE"

Downloading...
From (original): https://drive.google.com/uc?id=1N_-vMLwN0t6m4PjEdFgY7VySCiXwC1qE
From (redirected): https://drive.google.com/uc?id=1N_-vMLwN0t6m4PjEdFgY7VySCiXwC1qE&confirm=t&uuid=37eebbd2-241f-4a64-bd85-bd23f8860ebf
To: /content/GCF_013141755.1_UCI_ANSTEP_V1.0_genomic.fna.Z-SCORE
100% 277M/277M [00:05<00:00, 54.0MB/s]


In [63]:
df = pd.read_csv(
    'GCF_013141755.1_UCI_ANSTEP_V1.0_genomic.fna.Z-SCORE',
              skiprows=1,
              names=["start","end","3","4","5","score","seq","8"],
              delim_whitespace=True,
              chunksize=10000)

filtered_df = pd.concat(df, ignore_index=True)
filtered_df

/tmp/ipykernel_2658/953040244.py:1: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(


,start,end,3,4,5,score,seq,8
0,9331,9347,16,22.193,31.692,328.4535,gctacacgcgcaaagc,ASASASASASASSASA
1,9327,9343,16,22.202,30.342,325.3277,ctttgctacacgcgca,SASAASASASASASAS
2,9333,9349,16,22.195,31.585,327.5571,tacacgcgcaaagctg,ASASASASASASASAS
3,9329,9345,16,22.181,31.783,332.9764,ttgctacacgcgcaaa,SAASASASASASASAS
4,12099,12115,16,22.227,36.007,316.3602,ttgctgcacgcacaaa,SAASASASASASASAS
...,...,...,...,...,...,...,...,...
2953153,243470464,243470480,16,21.833,39.285,495.5430,tctcacgcacacacgg,SASASASASASASASA
2953154,243471088,243471112,24,20.752,34.724,1877.7650,ccggtggagactcgcgcacgcccg,ASASASASASASASASASASASAS
2953155,243471096,243471112,16,19.911,44.729,5864.6640,gactcgcgcacgcccg,ASASASASASASASAS
2953156,243473456,243473480,24,20.225,51.094,3794.7410,gggttacacacacgtgtgtgtggg,SASAASASASASASASASASASAS


In [64]:
zhunt_df = filtered_df[filtered_df['score'] > 400].reset_index(drop=True)
zhunt_df

,start,end,3,4,5,score,seq,8
0,30244,30260,16,21.625,41.959,632.7099,cgcgatgcgcctgcgc,ASSASASASASASASA
1,30248,30266,18,21.951,38.769,432.3963,atgcgcctgcgctgaagt,SASASASASASASASASA
2,30242,30260,18,21.943,38.944,436.3609,atcgcgatgcgcctgcgc,SAASSASASASASASASA
3,39388,39406,18,21.072,43.306,1245.8390,gaatttgtgcgcatgtgc,SASASASASASASASASA
4,39387,39409,22,21.046,47.065,1287.7860,tgaatttgtgcgcatgtgcctg,ASASASASASASASASASASAS
...,...,...,...,...,...,...,...,...
2455105,243470464,243470480,16,21.833,39.285,495.5430,tctcacgcacacacgg,SASASASASASASASA
2455106,243471088,243471112,24,20.752,34.724,1877.7650,ccggtggagactcgcgcacgcccg,ASASASASASASASASASASASAS
2455107,243471096,243471112,16,19.911,44.729,5864.6640,gactcgcgcacgcccg,ASASASASASASASAS
2455108,243473456,243473480,24,20.225,51.094,3794.7410,gggttacacacacgtgtgtgtggg,SASAASASASASASASASASASAS


In [65]:
zhunt_sorted = zhunt_df.sort_values(by=['start'])
zhunt_sorted[1] = 'genome'
zhunt_sorted[[1, 'start', 'end', 'score', 'seq']].to_csv('zscore_sorted.bed', sep='\t', header=None, index=False)

In [66]:
!bedtools merge -i zscore_sorted.bed > zscore_merged.bed

In [67]:
zhunt_coords = pd.read_csv('zscore_merged.bed', sep='\t', names=['place', 'start', 'end'])
zhunt_coords

,place,start,end
0,genome,30242,30266
1,genome,39382,39412
2,genome,46374,46392
3,genome,52912,52928
4,genome,57977,58005
...,...,...,...
218792,genome,243469860,243469893
218793,genome,243470455,243470485
218794,genome,243471087,243471118
218795,genome,243473277,243473293


скаффолды

In [68]:
region_df = annotation_df[annotation_df['type'] == 'region'].reset_index(drop=True)
region_df

,scaffold,source,type,start,end,score,strand,phase,attributes
0,NC_050201.1,RefSeq,region,1,22713616,.,+,.,ID=NC_050201.1:1..22713616;Dbxref=taxon:30069;...
1,NC_050202.1,RefSeq,region,1,93706023,.,+,.,ID=NC_050202.1:1..93706023;Dbxref=taxon:30069;...
2,NC_050203.1,RefSeq,region,1,88747589,.,+,.,ID=NC_050203.1:1..88747589;Dbxref=taxon:30069;...
3,NW_023404980.1,RefSeq,region,1,61276,.,+,.,ID=NW_023404980.1:1..61276;Dbxref=taxon:30069;...
4,NW_023404981.1,RefSeq,region,1,43775,.,+,.,ID=NW_023404981.1:1..43775;Dbxref=taxon:30069;...
...,...,...,...,...,...,...,...,...,...
489,NW_023405466.1,RefSeq,region,1,58513,.,+,.,ID=NW_023405466.1:1..58513;Dbxref=taxon:30069;...
490,NW_023405467.1,RefSeq,region,1,48590,.,+,.,ID=NW_023405467.1:1..48590;Dbxref=taxon:30069;...
491,NW_023405468.1,RefSeq,region,1,107798,.,+,.,ID=NW_023405468.1:1..107798;Dbxref=taxon:30069...
492,NW_023405469.1,RefSeq,region,1,61267,.,+,.,ID=NW_023405469.1:1..61267;Dbxref=taxon:30069;...


In [69]:
region_df['global_end'] = region_df['end'].cumsum()
region_df.head()

,scaffold,source,type,start,end,score,strand,phase,attributes,global_end
0,NC_050201.1,RefSeq,region,1,22713616,.,+,.,ID=NC_050201.1:1..22713616;Dbxref=taxon:30069;...,22713616
1,NC_050202.1,RefSeq,region,1,93706023,.,+,.,ID=NC_050202.1:1..93706023;Dbxref=taxon:30069;...,116419639
2,NC_050203.1,RefSeq,region,1,88747589,.,+,.,ID=NC_050203.1:1..88747589;Dbxref=taxon:30069;...,205167228
3,NW_023404980.1,RefSeq,region,1,61276,.,+,.,ID=NW_023404980.1:1..61276;Dbxref=taxon:30069;...,205228504
4,NW_023404981.1,RefSeq,region,1,43775,.,+,.,ID=NW_023404981.1:1..43775;Dbxref=taxon:30069;...,205272279


In [70]:
i_scaffold = 0
scaffold_start = 0
scaffold_end = 22713616
scaffold = 'NC_050201.1'

corrected_records = []

for index, row in zhunt_coords.iterrows():
    zdna_start = row['start']
    zdna_end = row['end']
    if zdna_start > scaffold_end:
        i_scaffold += 1
        if i_scaffold >= len(region_df):
            break

    if i_scaffold > 0:
        prev_cumulative_end = region_df.iloc[i_scaffold - 1]['global_end']
        scaffold_end = region_df.iloc[i_scaffold]['global_end']
        scaffold = region_df.iloc[i_scaffold]['scaffold']
    else:
        prev_cumulative_end = 0

    corrected_start = zdna_start - prev_cumulative_end
    corrected_end = zdna_end - prev_cumulative_end
    corrected_records.append({
            'scaffold': scaffold,
            'start': corrected_start,
            'end': corrected_end,
        })

corrected_zhunt_df = pd.DataFrame(corrected_records)
corrected_zhunt_df.head()

,scaffold,start,end
0,NC_050201.1,30242,30266
1,NC_050201.1,39382,39412
2,NC_050201.1,46374,46392
3,NC_050201.1,52912,52928
4,NC_050201.1,57977,58005


In [71]:
corrected_zhunt_df[[4, 5, 6]] = [4, 5, 6]
corrected_zhunt_df

,scaffold,start,end,4,5,6
0,NC_050201.1,30242,30266,4,5,6
1,NC_050201.1,39382,39412,4,5,6
2,NC_050201.1,46374,46392,4,5,6
3,NC_050201.1,52912,52928,4,5,6
4,NC_050201.1,57977,58005,4,5,6
...,...,...,...,...,...,...
218771,NW_023405470.1,72079,72100,4,5,6
218772,NW_023405470.1,72289,72305,4,5,6
218773,NW_023405470.1,72392,72427,4,5,6
218774,NW_023405470.1,74055,74086,4,5,6


In [72]:
corrected_zhunt_df.to_csv('zhunt.bed', sep='\t', header=None, index=False)

Результаты

In [73]:
!bedtools intersect -a zhunt.bed -b exons.bed -u > zhunt_in_exons.bed
!cat zhunt_in_exons.bed | wc -l

53675


In [74]:
!bedtools intersect -a zhunt.bed -b introns.bed -u > zhunt_in_introns.bed
!cat zhunt_in_introns.bed | wc -l

101047


In [75]:
!bedtools intersect -a zhunt.bed -b promoters.bed -u > zhunt_in_promoters.bed
!cat zhunt_in_promoters.bed | wc -l

13208


In [76]:
!bedtools intersect -a zhunt.bed -b downstream.bed -u > zhunt_in_downstream.bed
!cat zhunt_in_downstream.bed | wc -l

!bedtools intersect -a zhunt.bed -b intergenic.bed -u > zhunt_in_intergenic.bed
!cat zhunt_in_intergenic.bed | wc -l

!cat zhunt.bed | wc -l

2367
69180
218776


-------

In [77]:
!bedtools intersect -a exons.bed -b zhunt.bed -wa > exons_with_zhunt_all.bed
!sort -u exons_with_zhunt_all.bed > exons_with_zhunt.bed
!cat exons_with_zhunt.bed | wc -l
!cat exons.bed | wc -l

31691
229796


In [78]:
!bedtools intersect -a introns.bed -b zhunt.bed -wa > introns_with_zhunt_all.bed
!sort -u introns_with_zhunt_all.bed > introns_with_zhunt.bed
!cat introns_with_zhunt.bed | wc -l
!cat introns.bed | wc -l

18581
62211


In [79]:
!bedtools intersect -a promoters.bed -b zhunt.bed -wa > promoters_with_zhunt_all.bed
!sort -u promoters_with_zhunt_all.bed > promoters_with_zhunt.bed
!cat promoters_with_zhunt.bed | wc -l
!cat promoters.bed | wc -l

8262
15186


In [80]:
!bedtools intersect -a downstream.bed -b zhunt.bed -wa > downstream_with_zhunt_all.bed
!sort -u downstream_with_zhunt_all.bed > downstream_with_zhunt.bed
!cat downstream_with_zhunt.bed | wc -l
!cat downstream.bed | wc -l

2191
15187


In [81]:
!bedtools intersect -a intergenic.bed -b zhunt.bed -wa > intergenic_with_zhunt_all.bed
!sort -u intergenic_with_zhunt_all.bed > intergenic_with_zhunt.bed
!cat intergenic_with_zhunt.bed | wc -l
!cat intergenic.bed | wc -l

6584
12343


# 3. ZDNABERT

In [82]:
from Bio import SeqIO

# Название нужной хромосомы/скаффолда
target = "NC_050201.1"

# Если scaffolds уже существует
SeqIO.write(scaffolds[target], "NC_050201.1.fasta", "fasta")

1

In [83]:
rec = next(SeqIO.parse("NC_050201.1.fasta", "fasta"))
print(rec.id)
print(len(rec.seq))

NC_050201.1
22713616


In [84]:
!grep "^NC_050201.1" exons.bed > exons_chr.bed
!grep "^NC_050201.1" introns.bed > introns_chr.bed
!grep "^NC_050201.1" genes.bed > genes_chr.bed
!grep "^NC_050201.1" promoters.bed > promoters_chr.bed
!grep "^NC_050201.1" downstream.bed > downstream_chr.bed
!grep "^NC_050201.1" intergenic.bed > intergenic_chr.bed

In [85]:
!bedtools intersect -a zdnabert.bed -b exons_chr.bed -u > zdna_in_exons.bed
!bedtools intersect -a zdnabert.bed -b introns_chr.bed -u > zdna_in_introns.bed
!bedtools intersect -a zdnabert.bed -b promoters_chr.bed -u > zdna_in_promoters.bed
!bedtools intersect -a zdnabert.bed -b downstream_chr.bed -u > zdna_in_downstream.bed
!bedtools intersect -a zdnabert.bed -b intergenic_chr.bed -u > zdna_in_intergenic.bed

In [86]:
!wc -l zdna_in_exons.bed
!wc -l zdna_in_introns.bed
!wc -l zdna_in_promoters.bed
!wc -l zdna_in_downstream.bed
!wc -l zdna_in_intergenic.bed
!wc -l zdnabert.bed

3992 zdna_in_exons.bed
5027 zdna_in_introns.bed
826 zdna_in_promoters.bed
85 zdna_in_downstream.bed
3415 zdna_in_intergenic.bed
12266 zdnabert.bed


In [87]:
!bedtools intersect -a exons_chr.bed -b zdnabert.bed -wa > exons_with_zdna_all.bed
!sort -u exons_with_zdna_all.bed > exons_with_zdna.bed

!bedtools intersect -a introns_chr.bed -b zdnabert.bed -wa > introns_with_zdna_all.bed
!sort -u introns_with_zdna_all.bed > introns_with_zdna.bed

!bedtools intersect -a promoters_chr.bed -b zdnabert.bed -wa > promoters_with_zdna_all.bed
!sort -u promoters_with_zdna_all.bed > promoters_with_zdna.bed

!bedtools intersect -a downstream_chr.bed -b zdnabert.bed -wa > downstream_with_zdna_all.bed
!sort -u downstream_with_zdna_all.bed > downstream_with_zdna.bed

!bedtools intersect -a intergenic_chr.bed -b zdnabert.bed -wa > intergenic_with_zdna_all.bed
!sort -u intergenic_with_zdna_all.bed > intergenic_with_zdna.bed

In [88]:
!wc -l exons_with_zdna.bed
!wc -l exons_chr.bed

!wc -l introns_with_zdna.bed
!wc -l introns_chr.bed

!wc -l promoters_with_zdna.bed
!wc -l promoters_chr.bed

!wc -l downstream_with_zdna.bed
!wc -l downstream_chr.bed

!wc -l intergenic_with_zdna.bed
!wc -l intergenic_chr.bed

2568 exons_with_zdna.bed
24337 exons_chr.bed
1296 introns_with_zdna.bed
6625 introns_chr.bed
515 promoters_with_zdna.bed
1401 promoters_chr.bed
75 downstream_with_zdna.bed
1401 downstream_chr.bed
447 intergenic_with_zdna.bed
1092 intergenic_chr.bed


In [89]:
def wc(fname):
    with open(fname) as f:
        return sum(1 for _ in f)

total = wc("zdnabert.bed")

print("Exons:", wc("zdna_in_exons.bed") / total)
print("Introns:", wc("zdna_in_introns.bed") / total)
print("Promoters:", wc("zdna_in_promoters.bed") / total)
print("Downstream:", wc("zdna_in_downstream.bed") / total)
print("Intergenic:", wc("zdna_in_intergenic.bed") / total)

Exons: 0.325452470242948
Introns: 0.4098320560900049
Promoters: 0.06734061633784445
Downstream: 0.0069297244415457366
Intergenic: 0.27841187021033753


Квадруплексы и zhunt для одной хромосомы (для сравнения):

In [90]:
!grep "^NC_050201.1" quadruplex.bed > quadruplex_chr.bed
!grep "^NC_050201.1" zhunt.bed > zhunt_chr.bed

In [91]:
!wc -l quadruplex_chr.bed
!wc -l zhunt_chr.bed

2817 quadruplex_chr.bed
21874 zhunt_chr.bed


In [92]:
!bedtools intersect -a quadruplex_chr.bed -b exons_chr.bed -u > quadr_in_exons_chr.bed
!bedtools intersect -a quadruplex_chr.bed -b introns_chr.bed -u > quadr_in_introns_chr.bed
!bedtools intersect -a quadruplex_chr.bed -b promoters_chr.bed -u > quadr_in_promoters_chr.bed
!bedtools intersect -a quadruplex_chr.bed -b downstream_chr.bed -u > quadr_in_downstream_chr.bed
!bedtools intersect -a quadruplex_chr.bed -b intergenic_chr.bed -u > quadr_in_intergenic_chr.bed

In [93]:
!wc -l quadr_in_exons_chr.bed
!wc -l quadr_in_introns_chr.bed
!wc -l quadr_in_promoters_chr.bed
!wc -l quadr_in_downstream_chr.bed
!wc -l quadr_in_intergenic_chr.bed
!wc -l quadruplex_chr.bed

436 quadr_in_exons_chr.bed
1406 quadr_in_introns_chr.bed
148 quadr_in_promoters_chr.bed
30 quadr_in_downstream_chr.bed
1011 quadr_in_intergenic_chr.bed
2817 quadruplex_chr.bed


In [94]:
!bedtools intersect -a zhunt_chr.bed -b exons_chr.bed -u > zhunt_in_exons_chr.bed
!bedtools intersect -a zhunt_chr.bed -b introns_chr.bed -u > zhunt_in_introns_chr.bed
!bedtools intersect -a zhunt_chr.bed -b promoters_chr.bed -u > zhunt_in_promoters_chr.bed
!bedtools intersect -a zhunt_chr.bed -b downstream_chr.bed -u > zhunt_in_downstream_chr.bed
!bedtools intersect -a zhunt_chr.bed -b intergenic_chr.bed -u > zhunt_in_intergenic_chr.bed

In [95]:
!wc -l zhunt_in_exons_chr.bed
!wc -l zhunt_in_introns_chr.bed
!wc -l zhunt_in_promoters_chr.bed
!wc -l zhunt_in_downstream_chr.bed
!wc -l zhunt_in_intergenic_chr.bed
!wc -l zhunt_chr.bed

6639 zhunt_in_exons_chr.bed
9391 zhunt_in_introns_chr.bed
1422 zhunt_in_promoters_chr.bed
212 zhunt_in_downstream_chr.bed
6415 zhunt_in_intergenic_chr.bed
21874 zhunt_chr.bed


In [96]:
!bedtools intersect -a exons_chr.bed -b quadruplex_chr.bed -wa > exons_with_quadr_all.bed
!sort -u exons_with_quadr_all.bed > exons_with_quadr_chr.bed

!bedtools intersect -a introns_chr.bed -b quadruplex_chr.bed -wa > introns_with_quadr_all.bed
!sort -u introns_with_quadr_all.bed > introns_with_quadr_chr.bed

!bedtools intersect -a promoters_chr.bed -b quadruplex_chr.bed -wa > promoters_with_quadr_all.bed
!sort -u promoters_with_quadr_all.bed > promoters_with_quadr_chr.bed

!bedtools intersect -a downstream_chr.bed -b quadruplex_chr.bed -wa > downstream_with_quadr_all.bed
!sort -u downstream_with_quadr_all.bed > downstream_with_quadr_chr.bed

!bedtools intersect -a intergenic_chr.bed -b quadruplex_chr.bed -wa > intergenic_with_quadr_all.bed
!sort -u intergenic_with_quadr_all.bed > intergenic_with_quadr_chr.bed

In [97]:
!bedtools intersect -a exons_chr.bed -b zhunt_chr.bed -wa > exons_with_zhunt_all.bed
!sort -u exons_with_zhunt_all.bed > exons_with_zhunt_chr.bed

!bedtools intersect -a introns_chr.bed -b zhunt_chr.bed -wa > introns_with_zhunt_all.bed
!sort -u introns_with_zhunt_all.bed > introns_with_zhunt_chr.bed

!bedtools intersect -a promoters_chr.bed -b zhunt_chr.bed -wa > promoters_with_zhunt_all.bed
!sort -u promoters_with_zhunt_all.bed > promoters_with_zhunt_chr.bed

!bedtools intersect -a downstream_chr.bed -b zhunt_chr.bed -wa > downstream_with_zhunt_all.bed
!sort -u downstream_with_zhunt_all.bed > downstream_with_zhunt_chr.bed

!bedtools intersect -a intergenic_chr.bed -b zhunt_chr.bed -wa > intergenic_with_zhunt_all.bed
!sort -u intergenic_with_zhunt_all.bed > intergenic_with_zhunt_chr.bed

In [98]:
%%bash

echo "========== TOTAL =========="

echo "Quadruplex:"
wc -l quadruplex_chr.bed

echo "Zhunt:"
wc -l zhunt_chr.bed

echo "ZDNABERT:"
wc -l zdnabert.bed


echo
echo "========== TABLE 1 (number of predictions) =========="

echo "Quadruplex"
wc -l quadr_in_exons_chr.bed
wc -l quadr_in_introns_chr.bed
wc -l quadr_in_promoters_chr.bed
wc -l quadr_in_downstream_chr.bed
wc -l quadr_in_intergenic_chr.bed

echo
echo "Zhunt"
wc -l zhunt_in_exons_chr.bed
wc -l zhunt_in_introns_chr.bed
wc -l zhunt_in_promoters_chr.bed
wc -l zhunt_in_downstream_chr.bed
wc -l zhunt_in_intergenic_chr.bed

echo
echo "ZDNABERT"
wc -l zdna_in_exons.bed
wc -l zdna_in_introns.bed
wc -l zdna_in_promoters.bed
wc -l zdna_in_downstream.bed
wc -l zdna_in_intergenic.bed


echo
echo "========== TABLE 2 (regions with predictions) =========="

echo "TOTAL REGIONS"
wc -l exons_chr.bed
wc -l introns_chr.bed
wc -l promoters_chr.bed
wc -l downstream_chr.bed
wc -l intergenic_chr.bed

echo
echo "Quadruplex"
wc -l exons_with_quadr_chr.bed
wc -l introns_with_quadr_chr.bed
wc -l promoters_with_quadr_chr.bed
wc -l downstream_with_quadr_chr.bed
wc -l intergenic_with_quadr_chr.bed

echo
echo "Zhunt"
wc -l exons_with_zhunt_chr.bed
wc -l introns_with_zhunt_chr.bed
wc -l promoters_with_zhunt_chr.bed
wc -l downstream_with_zhunt_chr.bed
wc -l intergenic_with_zhunt_chr.bed

echo
echo "ZDNABERT"
wc -l exons_with_zdna.bed
wc -l introns_with_zdna.bed
wc -l promoters_with_zdna.bed
wc -l downstream_with_zdna.bed
wc -l intergenic_with_zdna.bed

========== TOTAL ==========
Quadruplex:
2817 quadruplex_chr.bed
Zhunt:
21874 zhunt_chr.bed
ZDNABERT:
12266 zdnabert.bed

========== TABLE 1 (number of predictions) ==========
Quadruplex
436 quadr_in_exons_chr.bed
1406 quadr_in_introns_chr.bed
148 quadr_in_promoters_chr.bed
30 quadr_in_downstream_chr.bed
1011 quadr_in_intergenic_chr.bed

Zhunt
6639 zhunt_in_exons_chr.bed
9391 zhunt_in_introns_chr.bed
1422 zhunt_in_promoters_chr.bed
212 zhunt_in_downstream_chr.bed
6415 zhunt_in_intergenic_chr.bed

ZDNABERT
3992 zdna_in_exons.bed
5027 zdna_in_introns.bed
826 zdna_in_promoters.bed
85 zdna_in_downstream.bed
3415 zdna_in_intergenic.bed

========== TABLE 2 (regions with predictions) ==========
TOTAL REGIONS
24337 exons_chr.bed
6625 introns_chr.bed
1401 promoters_chr.bed
1401 downstream_chr.bed
1092 intergenic_chr.bed

Quadruplex
385 exons_with_quadr_chr.bed
696 introns_with_quadr_chr.bed
136 promoters_with_quadr_chr.bed
29 downstream_with_quadr_chr.bed
292 intergenic_with_quadr_chr.bed

Zhunt

Фон

In [99]:
%%bash

echo "Exons"
awk '{sum += $3-$2} END {print sum}' exons.bed

echo "Introns"
awk '{sum += $3-$2} END {print sum}' introns.bed

echo "Promoters"
awk '{sum += $3-$2} END {print sum}' promoters.bed

echo "Downstream"
awk '{sum += $3-$2} END {print sum}' downstream.bed

echo "Intergenic"
awk '{sum += $3-$2} END {print sum}' intergenic.bed

Exons
142161798
Introns
111815113
Promoters
15161569
Downstream
3035982
Intergenic
92022024
